#**Preprocessing**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/AI Training/Datasets/Copy of shipping.csv')

In [ ]:
df.head()

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [ ]:
df.isnull().sum()

,0
ID,0
Warehouse_block,0
Mode_of_Shipment,0
Customer_care_calls,0
Customer_rating,0
Cost_of_the_Product,0
Prior_purchases,0
Product_importance,0
Gender,0
Discount_offered,0


In [ ]:
df['Reached.on.Time_Y.N'].value_counts()

,count
Reached.on.Time_Y.N,
1,6563
0,4436


In [ ]:
df.columns.to_list()

['ID',
 'Warehouse_block',
 'Mode_of_Shipment',
 'Customer_care_calls',
 'Customer_rating',
 'Cost_of_the_Product',
 'Prior_purchases',
 'Product_importance',
 'Gender',
 'Discount_offered',
 'Weight_in_gms',
 'Reached.on.Time_Y.N']

In [ ]:
x=df[['Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Gender', 'Discount_offered',  'Weight_in_gms']]
y=df['Reached.on.Time_Y.N']

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
y_test.value_counts()

,count
Reached.on.Time_Y.N,
1,1313
0,887


In [ ]:
1313/887

1.480270574971815

In [ ]:
y_train.value_counts()

,count
Reached.on.Time_Y.N,
1,5250
0,3549


In [ ]:
5250/3549

1.4792899408284024

In [ ]:
x_test.dtypes

,0
Warehouse_block,object
Mode_of_Shipment,object
Customer_care_calls,int64
Customer_rating,int64
Cost_of_the_Product,int64
Prior_purchases,int64
Product_importance,object
Gender,object
Discount_offered,int64
Weight_in_gms,int64


In [ ]:
from sklearn.preprocessing import LabelEncoder

cols=x_train.columns.to_list()
le=LabelEncoder()
for i in ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']:
  x_train[i]=le.fit_transform(x_train[i])
  x_test[i]=le.transform(x_test[i])

In [ ]:
x_train.head()

,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms
7920,3,1,4,1,261,2,2,0,9,4158
1529,4,2,3,3,162,2,1,0,26,1659
10521,1,2,3,2,221,2,2,1,9,4466
9558,3,2,3,5,157,4,2,0,2,4640
968,0,0,2,5,272,2,1,1,24,3638


In [ ]:
!pip install feature-engine
def remove_outlier(x_train, x_test):
  from feature_engine.outliers import Winsorizer
  winsor = Winsorizer(capping_method="quantiles",tail="both",fold=0.01)

  for col in x_train.columns.to_list():
    x_train[col] = winsor.fit_transform(x_train[[col]])
    x_test[col] = winsor.transform(x_test[[col]])

  return (x_train, x_test)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 5.4 MB/s eta 0:00:00


In [ ]:
x_train, x_test=remove_outlier(x_train, x_test)

In [ ]:
def scale_data(x_train, x_test):
  from sklearn.preprocessing import StandardScaler
  ss=StandardScaler()
  for i in x_train.columns.to_list():
    x_train[i]=ss.fit_transform(x_train[[i]])
    x_test[i]=ss.transform(x_test[[i]])

  return(x_train, x_test)

In [ ]:
x_train, x_test= scale_data(x_train, x_test)

In [ ]:
def feature_select(x_train, y_train):
  from sklearn.feature_selection import SelectKBest
  from sklearn.feature_selection import mutual_info_classif

  selector = SelectKBest(
      score_func=mutual_info_classif,
      k=5
  )

  x_train_new = selector.fit_transform(
      x_train,
      y_train
  )

  selected_features = x_train.columns[
      selector.get_support()
  ]

  return selected_features

In [ ]:
features=feature_select(x_train, y_train).to_list()

In [ ]:
features

['Mode_of_Shipment',
 'Customer_care_calls',
 'Prior_purchases',
 'Discount_offered',
 'Weight_in_gms']

In [ ]:
x_train=x_train[features]
x_test=x_test[features]

In [ ]:
x_test.head()

,Mode_of_Shipment,Customer_care_calls,Prior_purchases,Discount_offered,Weight_in_gms
1666,0.636601,-0.043121,-0.379737,2.832711,-0.948160
6542,0.636601,2.591731,0.272734,-0.704796,-1.107474
6428,-0.685120,-0.921405,-1.032208,-0.704796,1.353319
8060,-0.685120,0.835163,0.272734,-0.642734,0.402335
9822,0.636601,-0.921405,0.272734,-0.704796,1.012019


In [ ]:
def decompose(x_train, x_test):
  from sklearn.decomposition import PCA
  pca=PCA(n_components=2)
  x_train=pca.fit_transform(x_train)
  x_test=pca.transform(x_test)

  return (x_train, x_test)

In [ ]:
x_train, x_test=decompose(x_train, x_test)

In [ ]:
# def clear_imbalance(x_train, y_train):
#   from imblearn.over_sampling import RandomOverSampler
#   os=RandomOverSampler()

#   x_train_sampled, y_train_sampled = os.fit_resample(x_train, y_train)

#   return (x_train_sampled, y_train_sampled)


In [ ]:
# x_train, y_train=clear_imbalance(x_train, y_train)

In [ ]:
y_train.value_counts()

,count
Reached.on.Time_Y.N,
1,5250
0,3549


#**Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
model_dt=DecisionTreeClassifier(criterion='log_loss', max_depth=10, min_samples_split=3,  class_weight='balanced')

(*, criterion: str = "gini", splitter: str = "best", max_depth: Unknown | None = None, min_samples_split: int = 2, min_samples_leaf: int = 1, min_weight_fraction_leaf: float = 0, max_features: Unknown | None = None, random_state: Unknown | None = None, max_leaf_nodes: Unknown | None = None, min_impurity_decrease: float = 0, class_weight: Unknown | None = None, ccp_alpha: float = 0, monotonic_cst: Unknown | None = None) -> DecisionTreeClassifier

In [ ]:
model_dt.fit(x_train,y_train)

DecisionTreeClassifier(class_weight='balanced', criterion='log_loss',
                       max_depth=10, min_samples_split=3)

In [ ]:
y_pred=model_dt.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

print('accuracy_score')
a=accuracy_score(y_test, y_pred)
print(a)
'printconfusion_matrix'
print(confusion_matrix(y_test, y_pred))
print()
print('classification_report')
print(classification_report(y_test, y_pred))

accuracy_score
0.6609090909090909
[[831  56]
 [690 623]]

classification_report
              precision    recall  f1-score   support

           0       0.55      0.94      0.69       887
           1       0.92      0.47      0.63      1313

    accuracy                           0.66      2200
   macro avg       0.73      0.71      0.66      2200
weighted avg       0.77      0.66      0.65      2200



#**SVC**

In [ ]:
from sklearn.svm import SVC

model_svc=SVC(kernel='linear', class_weight='balanced')
model_svc.fit(x_train, y_train)
y_pred=model_svc.predict(x_test)

In [ ]:
# from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

print('accuracy_score')
a=accuracy_score(y_test, y_pred)
print(a)
'printconfusion_matrix'
print(confusion_matrix(y_test, y_pred))
print()
print('classification_report')
print(classification_report(y_test, y_pred))

accuracy_score
0.644090909090909
[[772 115]
 [668 645]]

classification_report
              precision    recall  f1-score   support

           0       0.54      0.87      0.66       887
           1       0.85      0.49      0.62      1313

    accuracy                           0.64      2200
   macro avg       0.69      0.68      0.64      2200
weighted avg       0.72      0.64      0.64      2200



#**Random Forest**


In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_rf=RandomForestClassifier(n_estimators=100, criterion='gini', max_depth=15, min_samples_split=3, bootstrap=False, class_weight='balanced')
model_rf.fit(x_train, y_train)
y_pred=model_rf.predict(x_test)

In [ ]:
# from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

print('accuracy_score')
a=accuracy_score(y_test, y_pred)
print(a)
'printconfusion_matrix'
print(confusion_matrix(y_test, y_pred))
print()
print('classification_report')
print(classification_report(y_test, y_pred))

accuracy_score
0.6604545454545454
[[757 130]
 [617 696]]

classification_report
              precision    recall  f1-score   support

           0       0.55      0.85      0.67       887
           1       0.84      0.53      0.65      1313

    accuracy                           0.66      2200
   macro avg       0.70      0.69      0.66      2200
weighted avg       0.73      0.66      0.66      2200



#**KNN**

In [ ]:
def clear_imbalance(x_train, y_train):
  from imblearn.over_sampling import RandomOverSampler
  os=RandomOverSampler()

  x_train_sampled, y_train_sampled = os.fit_resample(x_train, y_train)

  return (x_train_sampled, y_train_sampled)


In [ ]:
x_train, y_train=clear_imbalance(x_train, y_train)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
model_knn=KNeighborsClassifier(n_neighbors=16, weights='distance', algorithm='ball_tree', metric='minkowski',leaf_size=10)#leaf_size, ball_tree', 'kd_tree' 'brute'
model_knn.fit(x_train, y_train)
y_pred=model_knn.predict(x_test)

In [ ]:
# from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

print('accuracy_score')
a=accuracy_score(y_test, y_pred)
print(a)
'printconfusion_matrix'
print(confusion_matrix(y_test, y_pred))
print()
print('classification_report')
print(classification_report(y_test, y_pred))

accuracy_score
0.6522727272727272
[[679 208]
 [557 756]]

classification_report
              precision    recall  f1-score   support

           0       0.55      0.77      0.64       887
           1       0.78      0.58      0.66      1313

    accuracy                           0.65      2200
   macro avg       0.67      0.67      0.65      2200
weighted avg       0.69      0.65      0.65      2200



#**Naive bayes**

In [ ]:
from sklearn.naive_bayes import BernoulliNB

In [ ]:
model_bnb=BernoulliNB()
model_bnb.fit(x_train, y_train)
y_pred=model_bnb.predict(x_test)

In [ ]:
# from sklearn.metrics import accuracy_score, confusion_matrix,classification_report

print('accuracy_score')
a=accuracy_score(y_test, y_pred)
print(a)
'printconfusion_matrix'
print(confusion_matrix(y_test, y_pred))
print()
print('classification_report')
print(classification_report(y_test, y_pred))

accuracy_score
0.6359090909090909
[[ 387  500]
 [ 301 1012]]

classification_report
              precision    recall  f1-score   support

           0       0.56      0.44      0.49       887
           1       0.67      0.77      0.72      1313

    accuracy                           0.64      2200
   macro avg       0.62      0.60      0.60      2200
weighted avg       0.63      0.64      0.63      2200

